# Emote — emotion model pipeline

Thin wrapper over `pipeline/scripts/`: every active cell runs a script as a subprocess and skips work whose output already exists, unless its force flag (or the master switch `DO_RETRAIN`) is set. Display cells only read `results/` and `figures/`. Data lives in `$EMOTE_DATA` (default `~/emote_data`), never in the repo.

## 0. Setup and paths

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from IPython.display import Image, Markdown, display

PIPELINE = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve() / "pipeline"
EMOTE = PIPELINE.parent
SCRIPTS = PIPELINE / "scripts"
RESULTS, FIGURES = PIPELINE / "results", PIPELINE / "figures"
saved_root = PIPELINE / ".data_root"
DATA_ROOT = Path(os.environ.get("EMOTE_DATA") or (saved_root.read_text().strip() if saved_root.exists() else Path.home() / "emote_data"))
os.environ["EMOTE_DATA"] = str(DATA_ROOT)
os.environ.setdefault("HF_HOME", str(DATA_ROOT / "pretrained"))

def run(script, *args):
    cmd = [sys.executable, str(SCRIPTS / script), *map(str, args)]
    print("$", " ".join(cmd[1:]))
    r = subprocess.run(cmd)
    if r.returncode:
        raise RuntimeError(f"{script} failed with exit code {r.returncode}")

print("pipeline:", PIPELINE, "\ndata root:", DATA_ROOT)

## CONFIG — every knob in one place

In [ ]:
DO_RETRAIN = False                 # master switch: True re-runs everything below from scratch

FORCE_DOWNLOAD = DO_RETRAIN or False
FORCE_MANIFEST = DO_RETRAIN or False
FORCE_PROSODY  = DO_RETRAIN or False
FORCE_TRAIN    = DO_RETRAIN or False
FORCE_EXPORT   = DO_RETRAIN or False

SKIP_MELD   = False                # the 10.9 GB corpus with real transcripts
CREMA_LABEL = "acted"              # "acted" or "voice_vote" (what listeners heard)
GROUPS      = ["core"]             # configs/suite.yaml groups: core, extended, large
SEEDS       = [0, 1, 2]
OVERRIDES   = []                   # e.g. ["train.epochs=10"]

## 1. Inventory

In [ ]:
for d in ["ravdess", "crema_d", "tess", "meld", "iemocap", "msp_podcast"]:
    raw = DATA_ROOT / "raw" / d
    state = "downloaded" if (raw / "DOWNLOAD_OK.json").exists() else "present" if raw.is_dir() else "-"
    print(f"{d:12} {state}")
summary = DATA_ROOT / "manifests" / "summary.json"
if summary.exists():
    s = json.loads(summary.read_text())
    display(Markdown(f"**Manifest:** {s['clips']} clips, {s['hours']} h, speakers {s['speakers']}"))
runs = sorted(p.parent.relative_to(RESULTS) for p in RESULTS.glob("*/seed*/test_metrics.json"))
print(f"{len(runs)} finished run(s):", [str(r) for r in runs])

## 2. External downloads

In [ ]:
args = ["--skip-meld"] if SKIP_MELD else []
run("downloads/prepare_all_external.py", *args, *(["--force"] if FORCE_DOWNLOAD else []))

## 3. Compile data — manifest, 16 kHz cache, prosody features

In [ ]:
if FORCE_MANIFEST or not (DATA_ROOT / "manifests" / "manifest.csv").exists():
    run("preprocessing/build_manifest.py", "--crema-label", CREMA_LABEL, "--workers", os.cpu_count(), "--force")
run("preprocessing/extract_prosody.py", "--workers", os.cpu_count(), *(["--force"] if FORCE_PROSODY else []))

## 4. Quality checks — speaker leakage, missing clips, imbalance

In [ ]:
run("preprocessing/quality_checks.py")
q = json.loads((RESULTS / "quality_checks.json").read_text())
display(Markdown("\n".join(f"- {w}" for w in q["warnings"]) or "no warnings"))

## 4.5 Distributions

In [ ]:
run("visualization/analyze.py")
if (FIGURES / "fig01_dataset_overview.png").exists():
    display(Image(str(FIGURES / "fig01_dataset_overview.png")))

## 4.7 What flows into the model
- **audio tower**: 16 kHz waveform, loudness-normalised to −23 dBFS, random 6 s crop in training → WavLM-Base-Plus (layer-weighted) → attentive statistics pooling
- **text tower**: the corpus transcript in training, Whisper's transcript on the Jetson → emotion-pretrained DistilRoBERTa
- **prosody tower**: the 42 stage-2 features (pitch, energy, spectral, MFCC), z-scored with training statistics
- **targets**: 7 emotions masked per corpus label set; arousal / valence / dominance where the corpus has them

## 5. Train — experiment suite x seeds (each run is its own subprocess; finished runs are skipped)

In [ ]:
args = ["--groups", *GROUPS, "--seeds", *SEEDS]
if OVERRIDES:
    args += ["--override", *OVERRIDES]
if FORCE_TRAIN:
    args.append("--force")
run("training/train_suite.py", *args)

## 6. Analysis — tables, paired contrasts, figures

In [ ]:
run("visualization/analyze.py")
display(Markdown((RESULTS / "RESULTS.md").read_text()))
for f in sorted(FIGURES.glob("fig*.png")):
    print(f.name)
    display(Image(str(f)))

## 7. Export for the Jetson

In [ ]:
run("export/export_onnx.py", *(["--force"] if FORCE_EXPORT else []))
best = json.loads((RESULTS / "summary.json").read_text())["best_deployable"]
print("Copy", DATA_ROOT / "deploy" / f"{best}_seed0", "to the Jetson as Emote/emotion_deploy/ and run: bash go.sh emotions")